# 05 · Wearables: does it transfer from the bedside monitor?

Every result in notebooks 02–04 uses the GE monitor's own PPG and ECG. The patients also wore a **SmartCare
wrist PPG** and a **Faros/ePatch chest ECG**, the devices a ward or home deployment would actually use. This
notebook scores the same methods on those wearable signals, against the same impedance reference, over the
same 60-s windows.

* **Nothing is retrained.** Each patient's wearable windows are scored by the cross-validation model of the
  fold in which that patient was a test patient, so no model has seen the patient on either device.
* **The hard part is time.** The reference lives on the monitor's clock; the wearables keep their own, and
  none of the clocks can be trusted (section 1). Every file is aligned from the heart itself and only stretches
  whose alignment is verified beat by beat are used.

Built by `python -m data_prep.build_wearable` and `python -m scripts.wearable_eval all`; this notebook only
reads their outputs (and re-runs the alignment of two example files to show the evidence).

In [ ]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "estimate_rr").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, Markdown, display

BLUE, ORANGE, AQUA, INK, INK2, GRAY, VIOLET, RED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#c3c2b7", "#6250d6", "#e34948"
DEVCOL = {"monitor": INK2, "ppg": "#e87ba4", "ecg": VIOLET}       # devices get their own hues
DEVICE = {"ppg": "SmartCare wrist PPG", "ecg": "Faros/ePatch chest ECG"}
_js = False
def show(fig, height=None):
    """Render with plotly.js embedded once, so the exported page works offline."""
    global _js
    fig.update_layout(template="simple_white", height=height, margin=dict(l=60, r=24, t=70, b=90),
                      font=dict(family="system-ui, sans-serif", size=12, color=INK2), title_font=dict(size=15, color=INK),
                      legend=dict(orientation="h", yanchor="top", y=-0.18, x=0, xanchor="left"))
    display(HTML(fig.to_html(include_plotlyjs=not _js, full_html=False))); _js = True

W, R = Path("dataset/01NVa_wearable"), Path("runs/wearable")
al = pd.read_csv(W / "alignment.csv")
summary = json.loads((R / "summary.json").read_text()) if (R / "summary.json").exists() else None
display(Markdown(f"{al['path'].nunique()} wearable files of monitored patients; "
                 + ("results loaded from `runs/wearable/summary.json`." if summary else "**results not computed yet.**")))

## 1 · Four clocks, none of them right

* **SmartCare**: the file name is the start time; samples are counted at a nominal 100 Hz.
* **Faros/ePatch**: the EDF header has a start time in about half the files; the rest say 1 January 2000 (the
  device clock was never set).
* **GE monitor**: its header epoch and wall clock already disagree by 7 h.

Alignment therefore uses the heart. Heart rate from the wearable and from the monitor's ECG is cross-correlated
over the whole recording, jointly over offset and clock drift. The best candidates are then checked in 5-min
blocks on beat-to-beat heart-rate variability (mostly respiratory sinus arrhythmia), which differs from minute to
minute and patient to patient. A stretch is accepted only if at least three blocks agree on one straight-line
time map to within 1 s; a device restart starts a new stretch.

In [ ]:
f = al.drop_duplicates("path").copy()
acc = al[al["status"] == "aligned"]
rows = []
for k in ("ppg", "ecg"):
    fk = f[f["kind"] == k]; ak = acc[acc["kind"] == k]
    rows.append({"device": DEVICE[k], "files": len(fk), "files ≥ 30 min": int((fk["duration_h"] >= 0.5).sum()),
                 "files aligned": ak["path"].nunique(), "patients aligned": ak["patient_id"].nunique(),
                 "hours recorded (≥ 30 min files)": fk.loc[fk["duration_h"] >= 0.5, "duration_h"].sum(),
                 "hours aligned": (ak["t_end"] - ak["t_start"]).sum() / 3600,
                 "median block r": ak["median_r"].median(), "median timing residual (s)": ak["resid_sd_s"].median()})
display(pd.DataFrame(rows).set_index("device").style.format({"hours recorded (≥ 30 min files)": "{:,.0f}", "hours aligned": "{:,.0f}",
                                                             "median block r": "{:.2f}", "median timing residual (s)": "{:.2f}"}))
display(Markdown("Why files fail: " + ", ".join(f"{n} {s}" for s, n in f["status"].value_counts().items() if s != "aligned")
                 + ". \"Not aligned\" means no candidate passed the beat-level check, typically little overlap with the "
                 "monitor or a wrist signal too disturbed by movement; those files are left out rather than guessed."))

In [ ]:
a1 = acc.drop_duplicates("path")
fig = make_subplots(1, 2, subplot_titles=("Monitor clock minus device clock", "Device sample clock vs monitor"), horizontal_spacing=0.12)
for k in ("ppg", "ecg"):
    v = a1.loc[a1["kind"] == k, "offset_minus_nominal_h"].dropna()
    fig.add_trace(go.Histogram(x=v, xbins=dict(start=-12, end=36, size=0.25), name=f"{DEVICE[k]} ({len(v)} files with a clock)",
                               marker_color=DEVCOL[k], opacity=0.8), 1, 1)
    d = acc.loc[acc["kind"] == k, "drift_ppm"]
    fig.add_trace(go.Box(x=d, name=DEVICE[k], marker_color=DEVCOL[k], boxpoints="outliers", showlegend=False), 1, 2)
fig.update_xaxes(title_text="hours", row=1, col=1); fig.update_xaxes(title_text="drift (ppm; 1,000 ppm = 3.6 s per hour)", row=1, col=2)
fig.update_layout(barmode="overlay", title="No single correction would work: offsets cluster but differ, and the wrist PPG's clock drifts")
show(fig, 430)

### The evidence for one file of each device

Re-running the aligner on one example file per device: every dot is a block; its height is the lag at which the
wearable's beat-to-beat heart rate best matches the monitor's, its colour the correlation. True alignment shows up
as a straight line of well-correlated blocks (slope = clock drift); poorly correlated blocks (movement, ectopic
beats) scatter and are ignored.

In [ ]:
from data_prep.build_wearable import monitor_beats, monitor_segments, wearable_hr_1hz
from data_prep.wearable import read_smartcare, read_edf_ecg, beat_times, fine_align
segs = monitor_segments()
ex = {}
for k in ("ppg", "ecg"):
    cand = acc[(acc["kind"] == k) & (acc["duration_h"].between(6, 16))].sort_values("n_inliers", ascending=False)
    if cand.empty: continue
    p = cand.iloc[len(cand) // 3]                       # a typical well-aligned file, not the best
    rec = read_smartcare(p["path"]) if k == "ppg" else read_edf_ecg(p["path"])
    bw = beat_times(rec.signal, rec.fs, rec.kind)
    bm = monitor_beats(p["patient_id"], segs, W / "cache")
    v = fine_align(bw, bm, p["lag0_s"], p["drift_ppm"], coarse_search_s=60.0)
    ex[k] = (p, v["blocks"])
fig = make_subplots(1, len(ex), subplot_titles=[f"{DEVICE[k]} · {p['patient_id']}" for k, (p, _) in ex.items()], horizontal_spacing=0.1)
for j, (k, (p, b)) in enumerate(ex.items(), start=1):
    line = p["lag0_s"] + p["drift_ppm"] * 1e-6 * b["t"]
    fig.add_trace(go.Scatter(x=b["t"] / 3600, y=b["lag_s"] - p["lag0_s"], mode="markers",
                             marker=dict(color=b["r"], colorscale="Viridis", cmin=0, cmax=1, size=8, showscale=j == len(ex),
                                         colorbar=dict(title="block r")),
                             hovertemplate="%{x:.2f} h: lag %{y:.2f} s, r %{marker.color:.2f}<extra></extra>", showlegend=False), 1, j)
    fig.add_trace(go.Scatter(x=b["t"] / 3600, y=line - p["lag0_s"], mode="lines", line=dict(color=RED, width=1.5),
                             name="accepted time map" if j == 1 else None, showlegend=j == 1), 1, j)
    fig.update_xaxes(title_text="hours into the wearable recording", row=1, col=j)
fig.update_yaxes(title_text="lag minus offset (s)", row=1, col=1)
fig.update_layout(title="Block lags fall on one line (the slope is the clock drift)")
show(fig, 460)
for k, (p, _) in ex.items():
    display(Markdown(f"**{DEVICE[k]}**: offset {p['offset_minus_nominal_h']:+.2f} h from its own clock, drift {p['drift_ppm']:+.0f} ppm "
                     f"({abs(p['drift_ppm']) * 3.6e-3:.1f} s per hour), {p['n_inliers']} agreeing blocks, median r {p['median_r']:.2f}, "
                     f"residual {p['resid_sd_s']:.2f} s."))

### Do the beats line up?

The strictest check: after alignment, each wearable beat should sit next to a monitor beat. Below, for random
windows usable on both devices, the median time from each wearable beat to the nearest monitor beat. A correct
alignment gives a narrow peak at a small constant delay (pulse transit to the wrist, device filters); a wrong one
spreads over ±half a heartbeat.

In [ ]:
from preprocess.beats import detect_ppg_beats, detect_ecg_rpeaks
from data_prep.build_dataset import TARGET_FS
fig = go.Figure()
for k in ("ecg", "ppg"):
    wp = W / f"windows_{k}_w60_s30.parquet"
    if not wp.exists(): continue
    w = pd.read_parquet(wp); w = w[w[f"{k}_ok"] & w["monitor_ok"]]
    w = w.groupby("patient_id", group_keys=False).apply(lambda g: g.sample(min(len(g), 4), random_state=0))
    det, fs, d = (detect_ppg_beats if k == "ppg" else detect_ecg_rpeaks), TARGET_FS[k], []
    for seg, g in w.groupby("segment_id"):
        z = np.load(W / "signals" / f"{seg}.npz")
        for _, r in g.iterrows():
            a, b = int(r.start_s * fs), int((r.start_s + 60) * fs)
            bw, bm = det(z[k][a:b].astype(float), fs)[0] / fs, det(z[f"monitor_{k}"][a:b].astype(float), fs)[0] / fs
            if len(bw) < 20 or len(bm) < 20: continue
            j = np.clip(np.searchsorted(bm, bw), 1, len(bm) - 1)
            near = np.where(np.abs(bm[j] - bw) < np.abs(bm[j - 1] - bw), bm[j], bm[j - 1])
            d.append(np.median(bw - near) * 1000)
    d = np.array(d)
    fig.add_trace(go.Histogram(x=d, xbins=dict(start=-500, end=500, size=20), name=f"{DEVICE[k]}: median {np.median(d):+.0f} ms, "
                               f"IQR {np.subtract(*np.percentile(d, [75, 25])):.0f} ms ({len(d)} windows)", marker_color=DEVCOL[k], opacity=0.75))
fig.update_layout(barmode="overlay", title="Wearable beat minus nearest monitor beat (per window, ms)", xaxis_title="ms", yaxis_title="windows")
show(fig, 420)

## 2 · The same minute on two devices

One window usable on both devices: the monitor's signal, the wearable's, the impedance reference with its breaths,
and every estimate from each input.

In [ ]:
from estimate_rr.resp_signals import respiratory_waveform, RESP_FS
from estimate_rr.time_domain import count_adv
def example(k, seed):
    wp = W / f"windows_{k}_w60_s30.parquet"
    w = pd.read_parquet(wp).reset_index().rename(columns={"index": "window_row"})
    w = w[w[f"{k}_ok"] & w["monitor_ok"] & w["ref_ok"]]
    pr = pd.read_parquet(R / f"predictions_{k}.parquet") if (R / f"predictions_{k}.parquet").exists() else None
    mp = pd.read_parquet(R / f"monitor_predictions_paired_{k}.parquet") if (R / f"monitor_predictions_paired_{k}.parquet").exists() else None
    if pr is not None:
        w = w[w["window_row"].isin(pr.index) & w["window_row"].isin(mp.index)]
    r = w.sample(1, random_state=seed).iloc[0]
    z = np.load(W / "signals" / f"{r['segment_id']}.npz"); fs = TARGET_FS[k]
    a, b = int(r["start_s"] * fs), int((r["start_s"] + 60) * fs)
    t = np.arange(b - a) / fs
    resp = respiratory_waveform(z["resp"][int(r["start_s"] * 25):int((r["start_s"] + 60) * 25)].astype(float), 25.0)
    _, br = count_adv(resp, RESP_FS)
    fig = make_subplots(3, 1, shared_xaxes=True, vertical_spacing=0.07,
                        subplot_titles=(f"Monitor {k.upper()}", f"{DEVICE[k]} (aligned)", f"Impedance reference: {r['ref_rr']:.1f} breaths/min"))
    nz = lambda v: (v - np.median(v)) / (np.percentile(v, 99) - np.percentile(v, 1) + 1e-9)
    fig.add_trace(go.Scattergl(x=t, y=nz(z[f"monitor_{k}"][a:b].astype(float)), line=dict(color=DEVCOL["monitor"], width=1), showlegend=False), 1, 1)
    fig.add_trace(go.Scattergl(x=t, y=nz(z[k][a:b].astype(float)), line=dict(color=DEVCOL[k], width=1), showlegend=False), 2, 1)
    tr = np.arange(len(resp)) / RESP_FS
    fig.add_trace(go.Scatter(x=tr, y=resp, line=dict(color=INK, width=1.3), showlegend=False), 3, 1)
    fig.add_trace(go.Scatter(x=tr[br], y=resp[br], mode="markers", marker=dict(color=ORANGE, size=8), name="breaths"), 3, 1)
    fig.update_xaxes(title_text="seconds", row=3, col=1); fig.update_yaxes(showticklabels=False)
    fig.update_layout(title=f"{r['patient_id']} · the same 60 s on both devices")
    show(fig, 620)
    if pr is not None:
        tab = pd.DataFrame({"monitor input": mp.loc[r["window_row"]].drop(["patient_id", "ref_rr"], errors="ignore"),
                            "wearable input": pr.loc[r["window_row"], [c for c in mp.columns if c not in ("patient_id", "ref_rr")]]})
        tab["reference"] = r["ref_rr"]
        display(tab.astype(float).style.format("{:.1f}", na_rep="rejected"))
for k, seed in (("ppg", 4), ("ecg", 4)):
    if (W / f"windows_{k}_w60_s30.parquet").exists():
        example(k, seed)

## 3 · Results

Two views per device. **All wearable windows**: every aligned window with a usable reference in which the
wearable passes the quality checks. **Like for like**: the subset where the monitor's own signal is usable too,
scored with each input, so the difference is the device and nothing else.

In [ ]:
if summary is None:
    display(Markdown("Run `python -m scripts.wearable_eval all` first."))
else:
    for k, s in summary["signals"].items():
        display(Markdown(f"### {DEVICE[k]}\n{s['n_windows_worn']:,} aligned windows with a usable reference ({s['n_patients']} patients); "
                         f"the wearable passes quality in **{s['wearable_ok']:.0%}**, the monitor's {k.upper()} in {s['monitor_ok']:.0%}."))
        t = pd.DataFrame(s["all"]).set_index("estimator")[["coverage", "mae", "patient_macro_mae", "ci_lo", "ci_hi", "bias", "within_2bpm", "n_patients"]]
        display(t.sort_values("patient_macro_mae").style.format({"coverage": "{:.0%}", "mae": "{:.2f}", "patient_macro_mae": "{:.2f}", "ci_lo": "{:.2f}",
                                                                 "ci_hi": "{:.2f}", "bias": "{:+.2f}", "within_2bpm": "{:.0%}", "n_patients": "{:.0f}"}))
        pr = pd.DataFrame([{"estimator": r["estimator"], "monitor input": r["monitor"]["patient_macro_mae"],
                            "wearable input": r["wearable"]["patient_macro_mae"],
                            "Δ": r["wearable"]["patient_macro_mae"] - r["monitor"]["patient_macro_mae"],
                            "patients worse on wearable": f"{r['wearable_worse_patients']}/{r['n_patients']}", "p": r["p"],
                            "r(monitor, wearable estimate)": r["pred_corr"]} for r in s["paired"]]).set_index("estimator")
        display(Markdown(f"Like for like, {s['paired_n']:,} windows (per-patient MAE, bpm):"))
        display(pr.sort_values("wearable input").style.format({"monitor input": "{:.2f}", "wearable input": "{:.2f}", "Δ": "{:+.2f}",
                                                                "p": "{:.2g}", "r(monitor, wearable estimate)": "{:.2f}"}))

In [ ]:
if summary is not None:
    fig = make_subplots(1, 2, subplot_titles=[DEVICE[k] for k in summary["signals"]], horizontal_spacing=0.1)
    for j, k in enumerate(summary["signals"], start=1):
        w = pd.read_parquet(R / f"predictions_{k}.parquet"); m = pd.read_parquet(R / f"monitor_predictions_paired_{k}.parquet")
        lim = 0
        for label, color in (("DL CNN", BLUE), ("BreathNet + coherent + HR prior + conv head", AQUA), ("ARM · smart fusion", ORANGE)):
            if label not in w or label not in m: continue
            jn = m[["patient_id", "ref_rr"]].assign(mon=m[label], wear=w.loc[m.index, label]).dropna()
            per = jn.assign(em=(jn.mon - jn.ref_rr).abs(), ew=(jn.wear - jn.ref_rr).abs()).groupby("patient_id")[["em", "ew"]].mean()
            fig.add_trace(go.Scatter(x=per.em, y=per.ew, mode="markers", name=label, legendgroup=label, showlegend=j == 1,
                                     marker=dict(color=color, size=7, opacity=0.75, line=dict(color="white", width=0.5)),
                                     text=per.index, hovertemplate="%{text}<br>monitor %{x:.2f}, wearable %{y:.2f} bpm<extra></extra>"), 1, j)
            lim = max(lim, float(np.nanpercentile(per.to_numpy(), 98)))
        fig.add_trace(go.Scatter(x=[0, lim * 1.1], y=[0, lim * 1.1], mode="lines", line=dict(color=INK2, dash="dash", width=1),
                                 showlegend=False, hoverinfo="skip"), 1, j)
        fig.update_xaxes(title_text="per-patient MAE, monitor input", range=[0, lim * 1.1], row=1, col=j)
        fig.update_yaxes(title_text="per-patient MAE, wearable input", range=[0, lim * 1.1], row=1, col=j)
    fig.update_layout(title="Per patient, same windows: above the diagonal = the wearable serves this patient worse")
    show(fig, 520)

In [ ]:
if summary is not None:
    RANGES = [(0, 12), (12, 20), (20, 30), (30, 40), (40, 100)]
    fig = make_subplots(1, 2, subplot_titles=[DEVICE[k] for k in summary["signals"]], horizontal_spacing=0.1)
    for j, k in enumerate(summary["signals"], start=1):
        w = pd.read_parquet(R / f"predictions_{k}.parquet")
        for label, color in (("DL CNN", BLUE), ("BreathNet + coherent + HR prior + conv head", AQUA), ("ARM · smart fusion", ORANGE)):
            if label not in w: continue
            vals = [((w[label] - w.ref_rr).abs()[(w.ref_rr >= lo) & (w.ref_rr < hi)]).mean() for lo, hi in RANGES]
            ns = [int(((w.ref_rr >= lo) & (w.ref_rr < hi)).sum()) for lo, hi in RANGES]
            fig.add_trace(go.Bar(x=[f"{lo}–{hi if hi < 100 else ''}<br>(n={n:,})" for (lo, hi), n in zip(RANGES, ns)], y=vals, name=label,
                                 marker_color=color, legendgroup=label, showlegend=j == 1), 1, j)
        fig.update_yaxes(title_text="MAE (bpm)", row=1, col=j)
    fig.update_layout(title="Wearable input: error by reference breathing rate", barmode="group")
    show(fig, 440)

## 4 · Limits

* **A verified subset.** Only stretches whose alignment passed the beat-level check are scored; files that never
  overlapped the monitor, or whose wrist signal was too disturbed to verify, are left out rather than guessed.
  The reported coverage of the wearable is therefore its coverage *during verified stretches*.
* **Same reference, same windows.** The reference windows are the monitor's; windows where the impedance failed
  its checks are excluded on both devices alike.
* **No wearable training.** These are monitor-trained models transferred as they are; fine-tuning on wearable
  windows (patient-wise) is the obvious next step and would likely close part of any gap.